In [ ]:
# 1. Устанавливаем зависимость
# - uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
# - uv add --dev huggingface-hub jupyter

# 2. Скопировали download и embedder и поместили в цифру 09-...ipynb
# 3. Запускаем uv run python download.py

In [ ]:
# ____ Q1. Embedding a query ____

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
question = 'How does approximate nearest neighbor search work?'

In [23]:
# Method ___ 1
v1 = model.encode(question)
v1[0]

# np.float32(-0.020582037)

np.float32(-0.020582037)

In [46]:
# Method ___ 2
from embed.embedder import Embedder

model = Embedder()

v1_embbed = model.encode(question)
v1_embbed[0]

# -0.02058203437252893.

Exception: Системе не удается найти указанный путь. (os error 3)

In [ ]:
# ____ Q2. Cosine similarity  ____

In [17]:
linkToLesson = "02-vector-search/lessons/07-sqlitesearch-vector.md"

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
  repo_owner = "DataTalksClub",
  repo_name = "llm-zoomcamp",
  commit_id = "8c1834d",
  allowed_extensions={"md"},
  filename_filter = lambda path: linkToLesson in path,
)

documents = [file.parse() for file in reader.read()]

print(len(documents))             # 1
print(documents[0]["filename"])   # 02-vector-search/lessons/07-sqlitesearch-vector.md

In [28]:
content = documents[0]["content"][:1000]

"# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn't embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we'll want an approximate\nmethod instead.\n\nWhat we've done so far is exact

In [31]:
v2 = model.encode(content)

v1.dot(v2)   # v2 = np.float32(0.46690187)

np.float32(0.46690187)

In [ ]:
# ____ Q3. Chunking and search by hand  ____

In [38]:
documents

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [40]:
texts = []

for doc in documents:
  text = doc["filename"]
  texts.append(text)

In [41]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/1 [00:00<?, ?it/s]

1

In [44]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

7


In [ ]:
# Embed all chunks

X = model.encode_batch([c['content'] for c in chunks])
scores = X.dot(v)

In [ ]:
# Find the highest scoring chunk

best_idx = scores.argmax()
print(chunks[best_idx]['filename'])

In [ ]:
# ____ Q4. Vector search with minsearch  ____

In [57]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

from embedder import Embedder
from minsearch import VectorSearch

import numpy as np

In [59]:
embedder = Embedder()

Exception: Системе не удается найти указанный путь. (os error 3)